# CRG + ADD merge

Merge cleaned `V_CRG_STUDENT_COURSE` rows with one ADD semester snapshot row per `student_id`, `degree_id`, and `part_id`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

JOIN_KEYS = ["student_id", "degree_id", "part_id"]


def find_project_root(start=None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml and data/.")


PROJECT_ROOT = find_project_root()
PREPROCESSED_DIR = PROJECT_ROOT / "data" / "preprocessed"

CRG_PATH = PREPROCESSED_DIR / "V_CRG_STUDENT_COURSE" / "clean_v_crg_student_course.parquet"
ADD_PATH = PREPROCESSED_DIR / "V_ADD_STUDENT_DEGREE_STATUS" / "clean_v_add_student_degree_status.parquet"

OUTPUT_DIR = PREPROCESSED_DIR / "merge"
OUTPUT_PATH = OUTPUT_DIR / "merge_crg_add.parquet"
UNMATCHED_PATH = OUTPUT_DIR / "merge_crg_add_unmatched_add_snapshot.csv"

for path in [CRG_PATH, ADD_PATH]:
    assert path.exists(), f"Missing input file: {path}"

print("Project root:", PROJECT_ROOT)
print("CRG input:", CRG_PATH)
print("ADD input:", ADD_PATH)
print("Merged output:", OUTPUT_PATH)
print("Unmatched audit output:", UNMATCHED_PATH)

In [ ]:
df_crg = pd.read_parquet(CRG_PATH)
df_add = pd.read_parquet(ADD_PATH)
df_acd=pd.read_parquet(r"D:\AI\Real projects\Academic_Advisor\data\preprocessed\V_ACD_DEGREE_COURSE\v_acd_degree_course.parquet")
print("CRG shape:", df_crg.shape)
print("ADD shape:", df_add.shape)

missing_crg_cols = [col for col in JOIN_KEYS if col not in df_crg.columns]
missing_add_cols = [col for col in JOIN_KEYS if col not in df_add.columns]

assert not missing_crg_cols, f"CRG is missing join columns: {missing_crg_cols}"
assert not missing_add_cols, f"ADD is missing join columns: {missing_add_cols}"

print("\nCRG join-key null counts:")
print(df_crg[JOIN_KEYS].isna().sum())

print("\nADD join-key null counts:")
print(df_add[JOIN_KEYS].isna().sum())

print("\nCRG join-key dtypes:")
print(df_crg[JOIN_KEYS].dtypes)

print("\nADD join-key dtypes:")
print(df_add[JOIN_KEYS].dtypes)

In [ ]:
df_add.columns

## Select ADD snapshot columns

These are semester-start features plus audit fields. CRG remains the row-grain table, so ADD must be unique on the join keys before merging.

In [ ]:
ADD_SNAPSHOT_COLS = [
    "student_status_id",
    "student_id",
    "degree_id",
    "part_id",
    'prev_gpa_points',
    'prev_gpa_percent',
    "start_agpa_points",
    "start_agpa_percent",
    'total_semesters',
    'start_total_in_courses',
    'start_total_in_credits',
    'semester_reg_credits',
    'semester_reg_courses',
    'total_pass_credits',
    'total_fail_credits',
    "reg_total_semesters",
    "start_level_id",
    "start_level_name_pl",
    "start_part_id",
    "finish_part_id",
    "finish_status",
]

missing_snapshot_cols = [col for col in ADD_SNAPSHOT_COLS if col not in df_add.columns]
assert not missing_snapshot_cols, f"ADD is missing snapshot columns: {missing_snapshot_cols}"

df_add_snapshot = df_add[ADD_SNAPSHOT_COLS].copy() ## to not edit the original dataframe 

overlap_cols = [col for col in df_add_snapshot.columns if col in df_crg.columns and col not in JOIN_KEYS]
print("ADD snapshot rows:", len(df_add_snapshot))
print("ADD unique JOIN_KEYS:", df_add_snapshot[JOIN_KEYS].drop_duplicates().shape[0])
print("ADD duplicated rows on JOIN_KEYS:", df_add_snapshot.duplicated(JOIN_KEYS, keep=False).sum())
print("ADD columns that will receive '_add_snapshot' suffix:", overlap_cols)

if df_add_snapshot.duplicated(JOIN_KEYS).any():
    display(
        df_add_snapshot.loc[df_add_snapshot.duplicated(JOIN_KEYS, keep=False)]
        .sort_values(JOIN_KEYS)
        .head(100)
    )

assert df_add_snapshot.duplicated(JOIN_KEYS).sum() == 0, "ADD is not unique on JOIN_KEYS."

## Pre-merge validation

In [ ]:
df_add_snapshot.columns

In [ ]:
# pass_credit_ratiofail_credit_ratio = total_fail_credits / (total_pass_credits + total_fail_credits)
# pass_credit_ratio = total_pass_credits / degree_credits_count
# in_credit_ratio = total_in_credits / degree_credits_count

In [ ]:
print("=" * 80)
print("PRE-MERGE VALIDATION")
print("=" * 80)

print("CRG rows before merge:", len(df_crg))
print("ADD snapshot rows:", len(df_add_snapshot))
print("ADD unique JOIN_KEYS:", df_add_snapshot[JOIN_KEYS].drop_duplicates().shape[0])

df_merge_test = df_crg.merge(
    df_add_snapshot,
    on=JOIN_KEYS,
    how="left",
    validate="many_to_one",
    indicator=True,
    suffixes=("", "_add_snapshot"),
)

print("\nRows after merge:", len(df_merge_test))
print("Row count changed:", len(df_merge_test) - len(df_crg))

assert len(df_merge_test) == len(df_crg), "Row count changed after merge. Investigate duplicate ADD keys."

print("\nMerge result:")
print(df_merge_test["_merge"].value_counts(dropna=False))

missing_snapshot_mask = df_merge_test["_merge"].eq("left_only")

print("\nMissing ADD snapshot rows:", missing_snapshot_mask.sum())
print("Missing ADD snapshot ratio:", missing_snapshot_mask.mean())

unmatched_snapshot_report = df_merge_test.loc[missing_snapshot_mask].copy()

print("\nUnmatched rows by part_id:")
display(unmatched_snapshot_report["part_id"].value_counts(dropna=False).head(30))

print("\nUnmatched rows by degree_id:")
display(unmatched_snapshot_report["degree_id"].value_counts(dropna=False).head(30))

if "register_status" in unmatched_snapshot_report.columns:
    print("\nUnmatched rows by register_status:")
    display(unmatched_snapshot_report["register_status"].value_counts(dropna=False))

finish_status_col = "finish_status" if "finish_status" in unmatched_snapshot_report.columns else "finish_status_crg"
if finish_status_col in unmatched_snapshot_report.columns:
    print("\nUnmatched rows by CRG finish_status:")
    display(unmatched_snapshot_report[finish_status_col].value_counts(dropna=False))

sample_cols = [
    "student_course_id",
    "student_id",
    "degree_id",
    "part_id",
    "course_id",
    "grade_id",
    "final_mark",
    "points",
    finish_status_col,
    "register_status",
]
sample_cols = [col for col in sample_cols if col in unmatched_snapshot_report.columns]

print("\nUnmatched sample:")
display(unmatched_snapshot_report[sample_cols].head(100))

## Save merged table

In [ ]:
df_crg_add = df_merge_test.copy()
df_crg_add["has_add_snapshot"] = df_crg_add["_merge"].eq("both")
df_crg_add = df_crg_add.drop(columns=["_merge"])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_crg_add.to_parquet(OUTPUT_PATH, index=False)

unmatched_export_cols = [
    "student_course_id",
    "student_id",
    "degree_id",
    "part_id",
    "course_id",
    "grade_id",
    "final_mark",
    "points",
    finish_status_col,
    "register_status",
]
unmatched_export_cols = [col for col in unmatched_export_cols if col in unmatched_snapshot_report.columns]
unmatched_snapshot_report[unmatched_export_cols].to_csv(UNMATCHED_PATH, index=False)

print("Saved merged parquet:", OUTPUT_PATH)
print("Saved unmatched audit CSV:", UNMATCHED_PATH)
print("Merged shape:", df_crg_add.shape)
print("Rows with ADD snapshot:", int(df_crg_add["has_add_snapshot"].sum()))
print("Rows missing ADD snapshot:", int((~df_crg_add["has_add_snapshot"]).sum()))

In [ ]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

In [ ]:
display(df_crg_add.head())

In [ ]:
df_crg_add.info()

In [ ]:
df_crg_add.columns

In [ ]:
# Enrich df_crg_add with ACD degree-course information without changing df_crg_add.
# Assumes df_crg_add and df_acd already exist in memory.

required_crg_cols = ["degree_id", "course_id"]
required_acd_cols = [
    "degree_course_id_key",
    "degree_id_key",
    "course_id_key",
    "requirement_type_id",
    "requirement_type_sl",
    "course_credits",
    "credits_count",
    "course_name_sl",
    "degree_name_sl",
]

missing_crg_cols = [col for col in required_crg_cols if col not in df_crg_add.columns]
missing_acd_cols = [col for col in required_acd_cols if col not in df_acd.columns]

if missing_crg_cols:
    raise KeyError(f"df_crg_add is missing required columns: {missing_crg_cols}")
if missing_acd_cols:
    raise KeyError(f"df_acd is missing required columns: {missing_acd_cols}")

df_crg_add_row_count = len(df_crg_add)

acd_pairs = (
    df_acd[["degree_id_key", "course_id_key"]]
    .rename(columns={"degree_id_key": "acd_degree_id", "course_id_key": "course_id"})
    .drop_duplicates()
)

resolution_rows = []

for course_id, crg_course_rows in df_crg_add[["course_id", "degree_id"]].drop_duplicates().groupby("course_id", dropna=False):
    crg_degree_ids = crg_course_rows["degree_id"].drop_duplicates().tolist()
    acd_degree_ids = (
        acd_pairs.loc[acd_pairs["course_id"].eq(course_id), "acd_degree_id"]
        .drop_duplicates()
        .tolist()
    )
    acd_degree_count = len(acd_degree_ids)
    acd_degree_id_set = set(acd_degree_ids)

    for crg_degree_id in crg_degree_ids:
        if acd_degree_count == 0:
            issue_type = "course_not_found_in_acd"
            resolved_acd_degree_id = pd.NA
            recommended_action = "keep_original_degree_id_no_acd_course_found"
        elif crg_degree_id in acd_degree_id_set:
            issue_type = "valid_degree_course_pair"
            resolved_acd_degree_id = crg_degree_id
            recommended_action = "keep_original_degree_id"
        elif acd_degree_count == 1:
            issue_type = "crg_degree_not_in_acd_but_course_has_one_acd_degree"
            resolved_acd_degree_id = acd_degree_ids[0]
            recommended_action = "use_unique_acd_degree_id_for_acd_merge_only"
        else:
            issue_type = "crg_degree_not_in_acd_and_course_has_multiple_acd_degrees"
            resolved_acd_degree_id = pd.NA
            recommended_action = "ambiguous_keep_original_degree_id"

        resolution_rows.append(
            {
                "course_id": course_id,
                "crg_degree_id": crg_degree_id,
                "acd_degree_ids": acd_degree_ids,
                "acd_degree_count": acd_degree_count,
                "issue_type": issue_type,
                "resolved_acd_degree_id": resolved_acd_degree_id,
                "recommended_action": recommended_action,
            }
        )

degree_course_resolution_report = pd.DataFrame(
    resolution_rows,
    columns=[
        "course_id",
        "crg_degree_id",
        "acd_degree_ids",
        "acd_degree_count",
        "issue_type",
        "resolved_acd_degree_id",
        "recommended_action",
    ],
)

report_key_cols = ["course_id", "crg_degree_id"]
report_duplicate_mask = degree_course_resolution_report.duplicated(report_key_cols, keep=False)
if report_duplicate_mask.any():
    display(degree_course_resolution_report.loc[report_duplicate_mask].sort_values(report_key_cols).head(50))
    raise ValueError("degree_course_resolution_report is not unique on course_id + crg_degree_id.")

df_crg_add_resolved = df_crg_add.merge(
    degree_course_resolution_report,
    left_on=["course_id", "degree_id"],
    right_on=["course_id", "crg_degree_id"],
    how="left",
    validate="many_to_one",
)

if len(df_crg_add_resolved) != df_crg_add_row_count:
    raise ValueError("Row count changed while merging ACD resolution report back to df_crg_add.")

df_crg_add_resolved["student_degree_id_original"] = df_crg_add_resolved["degree_id"]
df_crg_add_resolved["degree_id_for_acd_merge"] = df_crg_add_resolved["degree_id"].copy()

safe_fallback_mask = df_crg_add_resolved["issue_type"].eq("crg_degree_not_in_acd_but_course_has_one_acd_degree")
if safe_fallback_mask.any():
    fallback_degree_ids = df_crg_add_resolved.loc[safe_fallback_mask, "resolved_acd_degree_id"].astype(
        df_crg_add_resolved["degree_id_for_acd_merge"].dtype
    )
    df_crg_add_resolved.loc[safe_fallback_mask, "degree_id_for_acd_merge"] = fallback_degree_ids
df_crg_add_resolved["degree_id_corrected_for_acd"] = safe_fallback_mask

resolution_rule_by_issue_type = {
    "valid_degree_course_pair": "original_degree_id",
    "crg_degree_not_in_acd_but_course_has_one_acd_degree": "corrected_to_unique_acd_degree_for_course",
    "crg_degree_not_in_acd_and_course_has_multiple_acd_degrees": "ambiguous_multiple_acd_degrees_not_corrected",
    "course_not_found_in_acd": "course_not_found_in_acd",
}
df_crg_add_resolved["acd_resolution_rule"] = df_crg_add_resolved["issue_type"].map(resolution_rule_by_issue_type)

df_acd_selected = (
    df_acd[required_acd_cols]
    .rename(
        columns={
            "degree_course_id_key": "degree_course_id",
            "degree_id_key": "acd_degree_id",
            "course_id_key": "course_id",
            "course_credits": "acd_course_credits",
            "credits_count": "degree_requirement_credits_count",
            "course_name_sl": "acd_course_name_sl",
            "degree_name_sl": "acd_degree_name_sl",
        }
    )
    .copy()
)

acd_selected_key_cols = ["acd_degree_id", "course_id"]
acd_selected_duplicate_mask = df_acd_selected.duplicated(acd_selected_key_cols, keep=False)
if acd_selected_duplicate_mask.any():
    display(df_acd_selected.loc[acd_selected_duplicate_mask].sort_values(acd_selected_key_cols).head(50))
    raise ValueError("df_acd_selected is not unique on acd_degree_id + course_id.")

pre_acd_merge_row_count = len(df_crg_add_resolved)

dfcrg_acd_add = df_crg_add_resolved.merge(
    df_acd_selected,
    left_on=["degree_id_for_acd_merge", "course_id"],
    right_on=["acd_degree_id", "course_id"],
    how="left",
    validate="many_to_one",
    indicator="_acd_merge",
)

if len(dfcrg_acd_add) != pre_acd_merge_row_count:
    raise ValueError("Row count changed after ACD merge. Investigate duplicate ACD degree-course keys.")
if len(dfcrg_acd_add) != df_crg_add_row_count:
    raise ValueError("Final row count does not match df_crg_add row count.")

dfcrg_acd_add["has_degree_course_info"] = dfcrg_acd_add["_acd_merge"].eq("both")
dfcrg_acd_add["acd_match_type"] = "unmatched"

exact_match_mask = dfcrg_acd_add["has_degree_course_info"] & dfcrg_acd_add["issue_type"].eq("valid_degree_course_pair")
fallback_match_mask = dfcrg_acd_add["has_degree_course_info"] & dfcrg_acd_add["issue_type"].eq(
    "crg_degree_not_in_acd_but_course_has_one_acd_degree"
)
ambiguous_unmatched_mask = dfcrg_acd_add["issue_type"].eq("crg_degree_not_in_acd_and_course_has_multiple_acd_degrees")
course_not_found_mask = dfcrg_acd_add["issue_type"].eq("course_not_found_in_acd")

dfcrg_acd_add.loc[exact_match_mask, "acd_match_type"] = "exact_degree_course_match"
dfcrg_acd_add.loc[fallback_match_mask, "acd_match_type"] = "unique_course_degree_fallback"
dfcrg_acd_add.loc[ambiguous_unmatched_mask, "acd_match_type"] = "unmatched_ambiguous_multiple_acd_degrees"
dfcrg_acd_add.loc[course_not_found_mask, "acd_match_type"] = "unmatched_course_not_found_in_acd"

ambiguous_acd_degree_report = dfcrg_acd_add.loc[ambiguous_unmatched_mask].copy()
course_not_found_in_acd_report = dfcrg_acd_add.loc[course_not_found_mask].copy()
unique_fallback_report = dfcrg_acd_add.loc[
    dfcrg_acd_add["issue_type"].eq("crg_degree_not_in_acd_but_course_has_one_acd_degree")
].copy()
unmatched_acd_after_resolution_report = dfcrg_acd_add.loc[dfcrg_acd_add["_acd_merge"].eq("left_only")].copy()

dfcrg_acd_add = dfcrg_acd_add.drop(columns=["_acd_merge"])

duplicated_student_course_id_count = "student_course_id column not found"
if "student_course_id" in dfcrg_acd_add.columns:
    duplicated_student_course_id_count = int(dfcrg_acd_add["student_course_id"].duplicated().sum())

print("=" * 80)
print("ACD ENRICHMENT VALIDATION SUMMARY")
print("=" * 80)
print("df_crg_add rows:", df_crg_add_row_count)
print("dfcrg_acd_add rows:", len(dfcrg_acd_add))
print("row difference:", len(dfcrg_acd_add) - df_crg_add_row_count)
print("\nissue_type value counts:")
print(dfcrg_acd_add["issue_type"].value_counts(dropna=False))
print("\nacd_resolution_rule value counts:")
print(dfcrg_acd_add["acd_resolution_rule"].value_counts(dropna=False))
print("\nacd_match_type value counts:")
print(dfcrg_acd_add["acd_match_type"].value_counts(dropna=False))
print("\nhas_degree_course_info value counts:")
print(dfcrg_acd_add["has_degree_course_info"].value_counts(dropna=False))
print("\nambiguous rows:", len(ambiguous_acd_degree_report))
print("course_not_found rows:", len(course_not_found_in_acd_report))
print("unmatched rows after final merge:", len(unmatched_acd_after_resolution_report))
print("duplicated student_course_id count:", duplicated_student_course_id_count)

In [ ]:
dfcrg_acd_add.head()

In [ ]:
dfcrg_acd_add.info()

In [ ]:
dfcrg_acd_add.to_parquet(r"D:\AI\Real projects\Academic_Advisor\data\features\merged_add_acd_crg.parquet")